In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd #to store data as csv or excel format
import time
from datetime import date

WRITE_FILE_PATH = r'c:\Interest_rate_files'
CHROME_DRIVER_PATH = 'c:\\Interest_rate_files\\chromedriver-win64\\chromedriver.exe'
URL = "https://www.cityunionbank.com/deposit-interest-rate" 

date1 = (date.today()).strftime('%d%m%Y') #Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' #Set directory path to write file

# Setup Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Provide the path to your ChromeDriver
service = Service(CHROME_DRIVER_PATH)

# Initialize the WebDriver
driver = webdriver.Chrome(executable_path=CHROME_DRIVER_PATH)

try:
    # Open the webpage
    driver.get(URL)

    # Wait until the content is loaded
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    # Give it extra time if needed
    time.sleep(5)

    # Get the page source after rendering JavaScript
    page_content = driver.page_source

    # You can parse this with BeautifulSoup if needed
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(page_content, 'html.parser')
    table=soup.find("table")
    rows=table.find_all("tr")
    
    print("Today's date =",date1)
    print("Interest rate file name = ",file_name)
    
    list_of_interest_rates = []
    tax_saver_row = []
    row_count = 0
    for i in rows[2:13]:  #Skip heading
        data = i.find_all("td")
        row=['City Union Bank']
        row.extend([tr.text for tr in data])
        row.insert(3,'')
        row.extend([''])
        if row_count == 10: row[1] = 'Tax-saver Fixed Deposit' 
            #for i in range(0,9):
            #    tax_saver_row[i] = ' '.join(str(tax_saver_row[i]).strip().replace('\n', '').split())
        list_of_interest_rates.append(row)
        row_count += 1
        
    #print(list_of_interest_rates)   
    pd.set_option('display.max_columns', None)    
    int_rate_df=pd.DataFrame(list_of_interest_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate',
                            'sr_rate','annualised_sr_rate'])
    print(int_rate_df)
    int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
    print("\nCity Union Bank = Success. Number of rows added = ", len(list_of_interest_rates))
finally:
    # Close the driver
    driver.quit()

Today's date = 19112024
Interest rate file name =  c:\Interest_rate_files\Interest_rate_v2_19112024.csv
          bank_name                       tenure gen_rate annualised_gen_rate  \
0   City Union Bank            7 days to 14 days    5.00%                       
1   City Union Bank           15 days to 45 days    5.50%                       
2   City Union Bank           46 days to 90 days    5.75%                       
3   City Union Bank          91 days to 180 days    6.00%                       
4   City Union Bank         181 days to 270 days    6.25%                       
5   City Union Bank         271 days to 332 days    6.50%                       
6   City Union Bank                     333 days    7.50%                       
7   City Union Bank         334 days to 400 days    7.00%                       
8   City Union Bank          401 days to 3 Years    6.50%                       
9   City Union Bank  Above 3 Years upto 10 Years    6.25%                       
10  C